# 02.1 KV Caching: Memory, Growth & Optimization

Deep-dive into KV cache mechanics: size calculations from first principles, memory pressure under real workloads, eviction strategies, and benchmarking generation speed with vs without caching.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from content.utils.kv_efficiency import kv_cache_size_gib
from content.utils.gpu_info import detect_gpu, print_gpu_info, GPU_CATALOG
from content.utils.benchmark import time_cuda

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    gpu = detect_gpu()
    print_gpu_info(gpu)

## 1. KV Cache Size Calculation from First Principles

$$\text{KV\_bytes} = 2 \times L \times h_{kv} \times d_h \times s \times b \times \text{dtype\_bytes}$$

- Factor 2 = K + V tensors
- $L$ = layers, $h_{kv}$ = KV heads, $d_h$ = head dim, $s$ = seq len, $b$ = batch

In [ ]:
def kv_cache_bytes(n_layers, n_kv_heads, head_dim, seq_len, batch=1, dtype_bytes=2):
    """Total KV cache in bytes."""
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * dtype_bytes

def fmt(b):
    """Format bytes to human-readable."""
    if b >= 1024**3: return f"{b/1024**3:.2f} GB"
    if b >= 1024**2: return f"{b/1024**2:.1f} MB"
    return f"{b/1024:.1f} KB"

# Model configs: (layers, kv_heads, head_dim)
MODELS = {
    'Llama 3 8B':   (32, 8, 128),
    'Llama 3 70B':  (80, 8, 128),
    'Llama 3 405B': (126, 8, 128),
    'Llama 2 7B (MHA)': (32, 32, 128),
}

print(f"{'Model':<20} {'1K ctx':>10} {'4K ctx':>10} {'32K ctx':>10} {'128K ctx':>10}")
print('-' * 65)
for name, (L, kv, hd) in MODELS.items():
    row = [fmt(kv_cache_bytes(L, kv, hd, s)) for s in [1024, 4096, 32768, 131072]]
    print(f"{name:<20} {row[0]:>10} {row[1]:>10} {row[2]:>10} {row[3]:>10}")

# Verify against utils
print(f"\nUtils check — Llama 3 70B @ 4K: {kv_cache_size_gib(80, 8, 128, 4096, 1):.3f} GiB")

## 2. Cache Growth with Sequence Length

KV cache grows **linearly** with sequence length. At long contexts, it dominates GPU memory.

In [ ]:
# Llama 3 70B: cache growth curves
L, kv, hd = 80, 8, 128
seq = np.arange(256, 131073, 256)

fig, ax = plt.subplots(figsize=(10, 5))
for batch in [1, 4, 16, 64]:
    gb = [kv_cache_bytes(L, kv, hd, int(s), batch) / 1024**3 for s in seq]
    ax.plot(seq / 1024, gb, label=f'batch={batch}', linewidth=2)

ax.axhline(45, color='red', ls='--', alpha=0.7, label='Available VRAM (~45GB)')
ax.set_xlabel('Sequence Length (K tokens)')
ax.set_ylabel('KV Cache (GB)')
ax.set_title('Llama 3 70B KV Cache Growth (GQA-8, FP16)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

per_tok = kv_cache_bytes(L, kv, hd, 1)
print(f"Per-token KV cost: {fmt(per_tok)} → at 128K context: {fmt(per_tok * 131072)}")

## 3. Memory Pressure Visualization

How much of an 80GB GPU is consumed by model weights vs KV cache at different batch/context combos?

In [ ]:
# Stacked bar: model weights + KV cache for various scenarios
model_weight_gb = 35  # Llama 3 70B FP16
gpu_vram = 80

scenarios = [
    ('B=1, 4K', 1, 4096), ('B=8, 4K', 8, 4096),
    ('B=32, 4K', 32, 4096), ('B=1, 32K', 1, 32768),
    ('B=4, 32K', 4, 32768), ('B=1, 128K', 1, 131072),
]

fig, ax = plt.subplots(figsize=(10, 5))
labels = [s[0] for s in scenarios]
kv_gb = [kv_cache_bytes(L, kv, hd, s[2], s[1]) / 1024**3 for s in scenarios]
x = range(len(scenarios))

ax.bar(x, [model_weight_gb]*len(scenarios), label='Model Weights', color='#3b82f6')
ax.bar(x, kv_gb, bottom=[model_weight_gb]*len(scenarios), label='KV Cache', color='#f97316')
ax.axhline(gpu_vram, color='red', ls='--', lw=2, label='80GB VRAM limit')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Memory (GB)')
ax.set_title('GPU Memory Pressure: Llama 3 70B (FP16) on A100-80GB')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, v in enumerate(kv_gb):
    total = model_weight_gb + v
    color = 'red' if total > gpu_vram else 'black'
    ax.text(i, total + 1, f'{total:.1f}', ha='center', fontsize=9, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: batch x seq_len -> cache GB
batches = [1, 2, 4, 8, 16, 32, 64]
seqs = [512, 1024, 2048, 4096, 8192, 16384, 32768]
grid = np.array([[kv_cache_bytes(L, kv, hd, s, b)/1024**3 for s in seqs] for b in batches])

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(seqs)))
ax.set_xticklabels([f'{s//1024}K' if s>=1024 else str(s) for s in seqs])
ax.set_yticks(range(len(batches)))
ax.set_yticklabels(batches)
ax.set_xlabel('Sequence Length'); ax.set_ylabel('Batch Size')
ax.set_title('KV Cache (GB) — Llama 3 70B GQA-8')
for i in range(len(batches)):
    for j in range(len(seqs)):
        c = 'white' if grid[i,j] > grid.max()*0.6 else 'black'
        ax.text(j, i, f'{grid[i,j]:.1f}', ha='center', va='center', fontsize=8, color=c)
plt.colorbar(im, label='GB')
plt.tight_layout()
plt.show()

## 4. GQA vs MHA: The Cache Multiplier

GQA reduces KV cache by ratio $\frac{h_{kv}}{h_q}$. For Llama 3 70B: 8/64 = 8x reduction.

In [ ]:
configs = {'MHA (64)': 64, 'GQA-8': 8, 'GQA-4': 4, 'MQA (1)': 1}
seq_r = np.linspace(512, 32768, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for label, kv_h in configs.items():
    vals = [kv_cache_bytes(80, kv_h, 128, int(s)) / 1024**3 for s in seq_r]
    ax1.plot(seq_r/1024, vals, label=label, linewidth=2)
ax1.axhline(45, color='red', ls='--', alpha=0.7, label='VRAM budget')
ax1.set_xlabel('Seq Length (K)'); ax1.set_ylabel('KV Cache (GB)')
ax1.set_title('Cache Size by Attention Type'); ax1.legend(); ax1.grid(True, alpha=0.3)

# Max batch at 4K
max_b = [int(45 / (kv_cache_bytes(80, kv_h, 128, 4096) / 1024**3)) for kv_h in configs.values()]
bars = ax2.bar(configs.keys(), max_b, color=['#ef4444','#3b82f6','#8b5cf6','#10b981'])
ax2.set_ylabel('Max Concurrent Requests')
ax2.set_title('Max Batch @ 4K ctx (45GB budget)')
for bar, v in zip(bars, max_b):
    ax2.text(bar.get_x()+bar.get_width()/2, v+1, str(v), ha='center')
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Cache Eviction Strategies

When KV cache exceeds memory budget, we must evict entries. Common strategies:
- **FIFO**: Drop oldest tokens (sliding window)
- **LRU**: Drop least-recently-attended tokens
- **Attention-score based (H2O)**: Keep tokens with highest cumulative attention
- **Streaming LLM**: Keep initial + recent tokens (attention sink pattern)

In [ ]:
# Simulate eviction strategies on attention patterns
torch.manual_seed(42)
seq_len_sim = 512
n_heads_sim = 8
budget = 128  # keep only 128 of 512 tokens

# Synthetic attention scores (sink pattern: first few tokens get high attention)
attn = torch.softmax(torch.randn(n_heads_sim, seq_len_sim), dim=-1)
attn[:, :4] += 0.3  # attention sink on first 4 tokens
attn = attn / attn.sum(dim=-1, keepdim=True)
cumulative_attn = attn.sum(dim=0)  # aggregate across heads

def evict_fifo(seq_len, budget):
    """Keep last `budget` tokens."""
    return list(range(seq_len - budget, seq_len))

def evict_h2o(cumulative_attn, budget):
    """Keep tokens with highest cumulative attention (Heavy Hitter Oracle)."""
    return cumulative_attn.topk(budget).indices.sort().values.tolist()

def evict_streaming(seq_len, budget, sink_size=4):
    """Keep first `sink_size` + last `budget - sink_size` tokens."""
    recent = budget - sink_size
    return list(range(sink_size)) + list(range(seq_len - recent, seq_len))

kept_fifo = evict_fifo(seq_len_sim, budget)
kept_h2o = evict_h2o(cumulative_attn, budget)
kept_stream = evict_streaming(seq_len_sim, budget)

# Visualize which tokens each strategy retains
fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
strategies = [('Attention Scores', None), ('FIFO (sliding window)', kept_fifo),
              ('H2O (heavy hitters)', kept_h2o), ('StreamingLLM (sink+recent)', kept_stream)]

for ax, (name, kept) in zip(axes, strategies):
    if kept is None:
        ax.bar(range(seq_len_sim), cumulative_attn.numpy(), width=1, color='#3b82f6', alpha=0.7)
        ax.set_ylabel('Attn')
    else:
        mask = np.zeros(seq_len_sim)
        mask[kept] = 1
        ax.bar(range(seq_len_sim), mask, width=1, color='#10b981', alpha=0.8)
        ax.set_ylabel('Kept')
        ax.set_ylim(0, 1.2)
    ax.set_title(name, fontsize=10, loc='left')
    ax.set_xlim(0, seq_len_sim)

axes[-1].set_xlabel('Token Position')
fig.suptitle(f'Eviction Strategies (budget={budget}/{seq_len_sim} tokens)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Measure quality: what fraction of total attention mass is retained?
def retained_attention_mass(cumulative_attn, kept_indices):
    total = cumulative_attn.sum().item()
    kept_mass = cumulative_attn[kept_indices].sum().item()
    return kept_mass / total * 100

results = {
    'FIFO': retained_attention_mass(cumulative_attn, kept_fifo),
    'H2O': retained_attention_mass(cumulative_attn, kept_h2o),
    'StreamingLLM': retained_attention_mass(cumulative_attn, kept_stream),
    'Full cache': 100.0,
}

print(f"Attention mass retained (budget={budget}/{seq_len_sim}):")
for name, pct in sorted(results.items(), key=lambda x: -x[1]):
    bar = '█' * int(pct / 2)
    print(f"  {name:<14} {pct:5.1f}% {bar}")

print(f"\n→ H2O retains {results['H2O'] - results['FIFO']:.1f}% more attention mass than FIFO")
print(f"→ StreamingLLM captures sink tokens that FIFO misses")

## 6. Benchmark: Generation Speed With vs Without KV Cache

Without KV cache, each decode step recomputes attention over the full sequence (quadratic). With caching, we only compute the new token's Q against cached K/V (linear).

In [ ]:
# Minimal transformer layer for benchmarking
class SimpleAttentionLayer(torch.nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = torch.nn.Linear(d_model, d_model, bias=False)
        self.scale = self.head_dim ** -0.5

    def forward(self, x, kv_cache=None):
        B, S, D = x.shape
        qkv = self.qkv(x).reshape(B, S, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, S, D)
        return self.out(out), (k, v)

d_model = 512
layer = SimpleAttentionLayer(d_model, 8).to(device).half()
print(f"Benchmark layer: d_model={d_model}, 8 heads, device={device}")

In [ ]:
# Generate N tokens: with cache vs without cache (full recompute)
@torch.no_grad()
def generate_with_cache(layer, prompt_len=64, gen_len=128):
    """Incremental decode: 1 token at a time, appending to KV cache."""
    x = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)
    _, kv = layer(x)  # prefill
    for _ in range(gen_len):
        new_tok = torch.randn(1, 1, d_model, device=device, dtype=torch.float16)
        _, kv = layer(new_tok, kv_cache=kv)

@torch.no_grad()
def generate_without_cache(layer, prompt_len=64, gen_len=128):
    """Full recompute: re-run entire sequence each step (no cache)."""
    tokens = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)
    for i in range(gen_len):
        layer(tokens)  # full forward on growing sequence
        new_tok = torch.randn(1, 1, d_model, device=device, dtype=torch.float16)
        tokens = torch.cat([tokens, new_tok], dim=1)

# Benchmark across different generation lengths
gen_lengths = [32, 64, 128, 256]
cached_times, uncached_times = [], []

for gl in gen_lengths:
    t_cached = time_cuda(lambda: generate_with_cache(layer, 64, gl), warmup=2, iterations=3)
    t_uncached = time_cuda(lambda: generate_without_cache(layer, 64, gl), warmup=2, iterations=3)
    cached_times.append(t_cached)
    uncached_times.append(t_uncached)
    print(f"gen_len={gl:>3}: cached={t_cached:>7.1f}ms  uncached={t_uncached:>7.1f}ms  speedup={t_uncached/t_cached:.1f}x")

In [ ]:
# Visualize speedup
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = range(len(gen_lengths))
ax1.bar([i-0.15 for i in x], cached_times, 0.3, label='With KV Cache', color='#10b981')
ax1.bar([i+0.15 for i in x], uncached_times, 0.3, label='Without Cache', color='#ef4444')
ax1.set_xticks(x)
ax1.set_xticklabels(gen_lengths)
ax1.set_xlabel('Generation Length (tokens)')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Generation Latency: Cached vs Uncached')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

speedups = [u/c for u, c in zip(uncached_times, cached_times)]
ax2.plot(gen_lengths, speedups, 'o-', color='#3b82f6', linewidth=2, markersize=8)
ax2.set_xlabel('Generation Length (tokens)')
ax2.set_ylabel('Speedup (x)')
ax2.set_title('KV Cache Speedup Factor')
ax2.grid(True, alpha=0.3)
ax2.axhline(1, color='gray', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()
print(f"\nKV cache speedup grows with generation length (O(n) vs O(n²) attention)")
print(f"At 256 tokens: {speedups[-1]:.1f}x faster with caching")

In [ ]:
# Per-token latency breakdown: shows O(1) vs O(n) per step
@torch.no_grad()
def measure_per_token(layer, prompt_len=64, gen_len=128, use_cache=True):
    """Measure latency of each decode step."""
    latencies = []
    if use_cache:
        x = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)
        _, kv = layer(x)
        for _ in range(gen_len):
            tok = torch.randn(1, 1, d_model, device=device, dtype=torch.float16)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            _, kv = layer(tok, kv_cache=kv)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)
    else:
        tokens = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)
        for _ in range(gen_len):
            tok = torch.randn(1, 1, d_model, device=device, dtype=torch.float16)
            tokens = torch.cat([tokens, tok], dim=1)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            layer(tokens)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)
    return latencies

lat_cached = measure_per_token(layer, use_cache=True)
lat_uncached = measure_per_token(layer, use_cache=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lat_cached, label='With KV Cache (O(1) per step)', color='#10b981', linewidth=1.5)
ax.plot(lat_uncached, label='Without Cache (O(n) per step)', color='#ef4444', linewidth=1.5)
ax.set_xlabel('Decode Step')
ax.set_ylabel('Per-Token Latency (ms)')
ax.set_title('Per-Token Decode Latency: Constant vs Growing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Cached: {np.mean(lat_cached):.3f}ms/tok (flat) | Uncached: {np.mean(lat_uncached):.3f}ms/tok (growing)")

## 7. Capacity Planning Summary

| Insight | Impact |
|---------|--------|
| KV cache grows linearly with seq × batch | Primary serving memory bottleneck |
| GQA-8 → 8x cache reduction vs MHA | 8x more concurrent requests |
| Without cache: O(n²) decode | KV cache gives 10-100x speedup at long contexts |
| H2O eviction retains >95% attention mass at 4x compression | Near-lossless quality with bounded memory |
| 128K context on 70B: single request ≈ 10GB cache | Long-context serving needs quantized KV or offloading |

In [ ]:
# Final capacity planning table
print('=' * 65)
print('CAPACITY PLANNING: Llama 3 70B on A100-80GB (GQA-8, FP16)')
print('=' * 65)
print(f'Model weights: ~35GB | Available for KV: ~45GB')
print(f'Per-token KV cost: {fmt(kv_cache_bytes(80, 8, 128, 1))}')
print()
print(f'{"Context":>8} | {"Per-Req":>8} | {"Max Batch":>9} | Recommendation')
print('-' * 65)
for ctx in [2048, 4096, 8192, 16384, 32768, 131072]:
    gb = kv_cache_bytes(80, 8, 128, ctx) / 1024**3
    mb = int(45 / gb)
    rec = 'High throughput' if mb >= 32 else 'Moderate' if mb >= 8 else 'Use KV quantization or offload'
    print(f'{ctx:>8} | {gb:>6.2f}GB | {mb:>9} | {rec}')

if device == 'cuda':
    torch.cuda.empty_cache()